In [1]:
%pip install matplotlib


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from math import comb
from random import Random

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch, Rectangle
from IPython.display import HTML, display

SERVER_COUNT = 10
INITIAL_LOAD = 0.70
REQUEST_LOAD = 0.05
REQUESTS = 140
TRIALS = 100_000
SEED = 2026

initial_servers = set(Random(SEED).sample(range(SERVER_COUNT), 2))
initial_loads = [
    INITIAL_LOAD if server in initial_servers else 0.0
    for server in range(SERVER_COUNT)
]

print("Серверы с начальной загрузкой 70%:", initial_servers)

Серверы с начальной загрузкой 70%: {1, 5}


In [3]:
rng = Random(SEED + 1)
random_hits = 0
power_two_hits = 0

for _ in range(TRIALS):
    random_hits += rng.randrange(SERVER_COUNT) in initial_servers

    candidates = rng.sample(range(SERVER_COUNT), 2)
    chosen = min(candidates, key=initial_loads.__getitem__)
    power_two_hits += chosen in initial_servers

p_random = random_hits / TRIALS
p_power_two = power_two_hits / TRIALS
exact_random = len(initial_servers) / SERVER_COUNT
exact_power_two = comb(len(initial_servers), 2) / comb(SERVER_COUNT, 2)

print(f"random:    empirical={p_random:.5f}, exact={exact_random:.5f}")
print(f"power-two: empirical={p_power_two:.5f}, exact={exact_power_two:.5f}")

assert abs(p_random - exact_random) <= 0.005
assert abs(p_power_two - exact_power_two) <= 0.005

random:    empirical=0.19812, exact=0.20000
power-two: empirical=0.02253, exact=0.02222


In [4]:
def simulate_requests(strategy, seed):
    loads = initial_loads.copy()
    history = [loads.copy()]
    dropped_history = [0]
    dropped = 0
    rng = Random(seed)

    for _ in range(REQUESTS):
        if strategy == "random":
            chosen = rng.randrange(SERVER_COUNT)
        elif strategy == "power_two":
            candidates = rng.sample(range(SERVER_COUNT), 2)
            chosen = min(candidates, key=loads.__getitem__)
        else:
            raise ValueError("Unknown strategy")

        if loads[chosen] >= 1.0:
            dropped += 1
        else:
            loads[chosen] = min(1.0, loads[chosen] + REQUEST_LOAD)

        history.append(loads.copy())
        dropped_history.append(dropped)

    return history, dropped_history


random_history, random_dropped = simulate_requests("random", SEED + 2)
power_history, power_dropped = simulate_requests("power_two", SEED + 3)


def load_color(load):
    if load >= 1.0 - 1e-12:
        return "#d62728"  # красный: 100%
    if load >= 0.80 - 1e-12:
        return "#ff7f0e"  # оранжевый: 80–99%
    if load >= 0.50 - 1e-12:
        return "#f2c94c"  # жёлтый: 50–79%
    return "#90ee90"      # зеленый: менее 50%


fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))
panels = []

for axis, title in zip(
    axes,
    ("Равномерный случайный выбор", "Power of Two Choices"),
):
    axis.set_xlim(-0.15, 5.0)
    axis.set_ylim(-0.25, 2.25)
    axis.set_aspect("equal")
    axis.axis("off")
    axis.set_title(title, fontsize=13, fontweight="bold")

    fills = []
    percentages = []
    for server in range(SERVER_COUNT):
        x = server % 5
        y = 1 - server // 5
        axis.add_patch(Rectangle((x, y), 0.82, 0.82, fill=False,
                                 edgecolor="#333333", linewidth=1.5))
        fill = Rectangle((x, y), 0.82, 0.0, facecolor="#4c78a8")
        axis.add_patch(fill)
        fills.append(fill)
        percentages.append(
            axis.text(x + 0.41, y + 0.43, "0%", ha="center", va="center",
                      fontsize=10, fontweight="bold")
        )
        axis.text(x + 0.41, y - 0.08, f"S{server}", ha="center", va="top",
                  fontsize=9)

    status = axis.text(2.41, 2.08, "", ha="center", fontsize=10)
    panels.append((fills, percentages, status))

legend = [
    Patch(color="#90ee90", label="< 50%"),
    Patch(color="#f2c94c", label="50–79%"),
    Patch(color="#ff7f0e", label="80–99%"),
    Patch(color="#d62728", label="100%"),
]
fig.legend(handles=legend, loc="lower center", ncol=4, frameon=False)
heading = fig.suptitle("", fontsize=14)
fig.tight_layout(rect=(0, 0.08, 1, 0.92))


def update(frame):
    heading.set_text(f"Обработано запросов: {frame} из {REQUESTS}")
    states = (
        (random_history[frame], random_dropped[frame]),
        (power_history[frame], power_dropped[frame]),
    )

    for (fills, percentages, status), (loads, dropped) in zip(panels, states):
        for fill, label, load in zip(fills, percentages, loads):
            fill.set_height(0.82 * load)
            fill.set_facecolor(load_color(load))
            label.set_text(f"{load:.0%}")
        status.set_text(f"Отклонено запросов: {dropped}")


animation = FuncAnimation(
    fig,
    update,
    frames=range(REQUESTS + 1),
    interval=120,
    repeat=False,
)
plt.close(fig)
display(HTML(animation.to_jshtml()))